# Day 09. Exercise 01
# Gridsearch

## 0. Imports

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from tqdm.notebook import tqdm
from itertools import product
import joblib
import warnings
warnings.filterwarnings('ignore')

## 1. Preprocessing

1. Read the file [`day-of-week-not-scaled.csv`](https://drive.google.com/file/d/1AlGvsJDSzPT_70caausx8bFuupIEZkfh/view?usp=sharing). It is similar to the one from the previous exercise, but this time we did not scale continuous features (we are not going to use logreg anymore). Don't forget to enrich the table with the 'dayofweek' column from the previous day's .csv-file.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [2]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv')
df['dayofweek'] = pd.read_csv('../data/dayofweek.csv')['dayofweek']

print(df.shape)
df.head()

(1686, 44)


,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1,dayofweek
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4


In [3]:
X = df.drop(columns=['dayofweek'])
y = df['dayofweek']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")

X_train: (1348, 43), X_test: (338, 43)
y_train: (1348,), y_test: (338,)


## 2. SVM gridsearch

1. Using `GridSearchCV` try different parameters of kernel (`linear`, `rbf`, `sigmoid`), C (`0.01`, `0.1`, `1`, `1.5`, `5`, `10`), gamma (`scale`, `auto`), class_weight (`balanced`, `None`) use `random_state=21` and `probability=True` and get the best combination of them in terms of accuracy.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`. Check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [4]:
%%time
svm_params = {
    'kernel': ['linear', 'rbf', 'sigmoid'], 'C': [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma': ['scale', 'auto'], 'class_weight': ['balanced', None],
    'random_state': [21], 'probability': [True]
}

gs_svm = GridSearchCV(
    estimator=SVC(), param_grid=svm_params, scoring='accuracy',
    cv=5, n_jobs=8, verbose=1
)
gs_svm.fit(X_train, y_train)
print(f"\nЛучшие параметры: {gs_svm.best_params_}")
print(f"Лучшая accuracy в результате GridSearchCV: {gs_svm.best_score_}")

Fitting 5 folds for each of 72 candidates, totalling 360 fits

Лучшие параметры: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Лучшая accuracy в результате GridSearchCV: 0.8761090458488228
CPU times: user 1.19 s, sys: 113 ms, total: 1.3 s
Wall time: 2min 18s


In [5]:
svm_results = pd.DataFrame(gs_svm.cv_results_)
svm_results = svm_results.sort_values('rank_test_score', ascending=True)
svm_results[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']].head(10)

,params,mean_test_score,std_test_score,rank_test_score
70,"{'C': 10, 'class_weight': None, 'gamma': 'auto...",0.876109,0.018419,1
64,"{'C': 10, 'class_weight': 'balanced', 'gamma':...",0.863500,0.010870,2
58,"{'C': 5, 'class_weight': None, 'gamma': 'auto'...",0.816018,0.008116,3
52,"{'C': 5, 'class_weight': 'balanced', 'gamma': ...",0.808608,0.021007,4
60,"{'C': 10, 'class_weight': 'balanced', 'gamma':...",0.721052,0.034438,5
63,"{'C': 10, 'class_weight': 'balanced', 'gamma':...",0.721052,0.034438,5
66,"{'C': 10, 'class_weight': None, 'gamma': 'scal...",0.719587,0.017463,7
69,"{'C': 10, 'class_weight': None, 'gamma': 'auto...",0.719587,0.017463,7
51,"{'C': 5, 'class_weight': 'balanced', 'gamma': ...",0.706234,0.031619,9
48,"{'C': 5, 'class_weight': 'balanced', 'gamma': ...",0.706234,0.031619,9


## 3. Decision tree

1. Using `GridSearchCV` try different parameters of `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use `random_state=21`.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [6]:
%%time
tree_params = {
    'max_depth': list(range(1, 50)),
    'class_weight': ['balanced', None],
    'criterion': ['entropy', 'gini'],
    'random_state': [21]
}

gs_tree = GridSearchCV(
    estimator=DecisionTreeClassifier(), param_grid=tree_params, scoring='accuracy',
    cv=5, n_jobs=8, verbose=1
)
gs_tree.fit(X_train, y_train)
print(f"\nЛучшие параметры: {gs_tree.best_params_}")
print(f"Лучшая accuracy в результате GridSearchCV: {gs_tree.best_score_:.5f}")

Fitting 5 folds for each of 196 candidates, totalling 980 fits

Лучшие параметры: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 22, 'random_state': 21}
Лучшая accuracy в результате GridSearchCV: 0.87312
CPU times: user 447 ms, sys: 55.2 ms, total: 502 ms
Wall time: 951 ms


In [7]:
tree_results = pd.DataFrame(gs_tree.cv_results_)
tree_results = tree_results.sort_values('rank_test_score', ascending=True)
tree_results[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']].head(10)

,params,mean_test_score,std_test_score,rank_test_score
70,"{'class_weight': 'balanced', 'criterion': 'gin...",0.873121,0.023998,1
69,"{'class_weight': 'balanced', 'criterion': 'gin...",0.873121,0.026300,2
80,"{'class_weight': 'balanced', 'criterion': 'gin...",0.873116,0.023911,3
81,"{'class_weight': 'balanced', 'criterion': 'gin...",0.873116,0.023911,3
96,"{'class_weight': 'balanced', 'criterion': 'gin...",0.873116,0.023911,3
97,"{'class_weight': 'balanced', 'criterion': 'gin...",0.873116,0.023911,3
82,"{'class_weight': 'balanced', 'criterion': 'gin...",0.873116,0.023911,3
83,"{'class_weight': 'balanced', 'criterion': 'gin...",0.873116,0.023911,3
87,"{'class_weight': 'balanced', 'criterion': 'gin...",0.873116,0.023911,3
86,"{'class_weight': 'balanced', 'criterion': 'gin...",0.873116,0.023911,3


## 4. Random forest

1. Using `GridSearchCV` try different parameters of `n_estimators` (`5`, `10`, `50`, `100`), `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use random_state=21.
2. Create a dataframe from the results of the gridsearch and sort it ascendengly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [8]:
%%time
rf_params = {
    'n_estimators': [5, 10, 50, 100], 'max_depth': list(range(1, 50)),
    'class_weight': ['balanced', None], 'criterion': ['entropy', 'gini'],
    'random_state': [21]
}

gs_rf = GridSearchCV(
    estimator=RandomForestClassifier(), param_grid=rf_params, scoring='accuracy',
    cv=5, n_jobs=8, verbose=1
)
gs_rf.fit(X_train, y_train)
print(f"\nЛучшие параметры: {gs_rf.best_params_}")
print(f"Лучшая accuracy в результате GridSearchCV: {gs_rf.best_score_:.5f}")

Fitting 5 folds for each of 784 candidates, totalling 3920 fits

Лучшие параметры: {'class_weight': None, 'criterion': 'gini', 'max_depth': 28, 'n_estimators': 50, 'random_state': 21}
Лучшая accuracy в результате GridSearchCV: 0.90429
CPU times: user 1.76 s, sys: 203 ms, total: 1.96 s
Wall time: 27.4 s


In [9]:
rf_results = pd.DataFrame(gs_rf.cv_results_)
rf_results = rf_results.sort_values('rank_test_score', ascending=True)
rf_results[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']].head(10)

,params,mean_test_score,std_test_score,rank_test_score
698,"{'class_weight': None, 'criterion': 'gini', 'm...",0.904290,0.010961,1
711,"{'class_weight': None, 'criterion': 'gini', 'm...",0.903547,0.014380,2
314,"{'class_weight': 'balanced', 'criterion': 'gin...",0.902817,0.013554,3
330,"{'class_weight': 'balanced', 'criterion': 'gin...",0.902809,0.013010,4
735,"{'class_weight': None, 'criterion': 'gini', 'm...",0.902806,0.010460,5
739,"{'class_weight': None, 'criterion': 'gini', 'm...",0.902806,0.010460,5
775,"{'class_weight': None, 'criterion': 'gini', 'm...",0.902806,0.010460,5
783,"{'class_weight': None, 'criterion': 'gini', 'm...",0.902806,0.010460,5
702,"{'class_weight': None, 'criterion': 'gini', 'm...",0.902806,0.011698,5
743,"{'class_weight': None, 'criterion': 'gini', 'm...",0.902806,0.010460,5


## 5. Progress bar

Gridsearch can be a quite long process and you may find yourself wondering when it will end.
1. Create a manual gridsearch for the same parameters values of random forest iterating through the list of the possible values and calculating `cross_val_score` for each combination. Try to increase `n_jobs`. The value `cv` for `cross_val_score` is 5.
2. Track the progress using the library `tqdm.notebook`.
3. Create a dataframe from the results of the gridsearch with the columns corresponding to the names of the parameters and `mean_accuracy` and `std_accuracy`.
4. Sort it descendingly by the `mean_accuracy`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [10]:
%%time

n_estimators_list = [5, 10, 50, 100]
max_depth_list = list(range(1, 50))
class_weight_list = ['balanced', None]
criterion_list = ['entropy', 'gini']

combinations = list(product(n_estimators_list, max_depth_list, class_weight_list, criterion_list))
total = len(combinations)
print(f"Всего комбинаций: {total}")

results_list = []

for n_est, depth, cw, crit in tqdm(combinations, total=total, desc="RF GridSearch", leave=True, dynamic_ncols=True):
    rf = RandomForestClassifier(
        n_estimators=n_est, max_depth=depth, class_weight=cw,
        criterion=crit, random_state=21,
    )
    scores = cross_val_score(rf, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
    results_list.append({
        'n_estimators': n_est, 'max_depth': depth, 'class_weight': cw,
        'criterion': crit, 'mean_accuracy': scores.mean(), 'std_accuracy': scores.std()
    })

Всего комбинаций: 784


RF GridSearch:   0%|          | 0/784 [00:00<?, ?it/s]

CPU times: user 7.69 s, sys: 957 ms, total: 8.65 s
Wall time: 56.8 s


In [11]:
manual_results = pd.DataFrame(results_list)
manual_results = manual_results.sort_values('mean_accuracy', ascending=False)
manual_results.head(10)

,n_estimators,max_depth,class_weight,criterion,mean_accuracy,std_accuracy
503,50,28,NaN,gini,0.904290,0.010961
711,100,31,NaN,gini,0.903547,0.014380
509,50,30,balanced,gini,0.902817,0.013554
525,50,34,balanced,gini,0.902809,0.013010
751,100,41,NaN,gini,0.902806,0.010460
755,100,42,NaN,gini,0.902806,0.010460
759,100,43,NaN,gini,0.902806,0.010460
767,100,45,NaN,gini,0.902806,0.010460
747,100,40,NaN,gini,0.902806,0.010460
783,100,49,NaN,gini,0.902806,0.010460


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.

In [12]:
best_model = gs_rf.best_estimator_
y_pred = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Итоговая accuracy на тестовом датасете: {test_accuracy:.5f}")

Итоговая accuracy на тестовом датасете: 0.92899


In [13]:
joblib.dump(best_model, 'gridsearch_rf.sav')

['gridsearch_rf.sav']